In [1]:
import numpy as np
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.manual_seed(42)
np.random.seed(42)

In [2]:
class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)


class SinNoise(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=32, out_dim=1):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            Sine(),
            nn.Linear(hidden_dim, hidden_dim),
            Sine(),
            nn.Linear(hidden_dim, out_dim),
        )
        self._init_with_noise()

    def _init_with_noise(self, std=0.5):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)
                nn.init.normal_(module.bias, mean=0.0, std=std)

    def forward(self, x):
        return self.base(x)


class SinTanhNoise(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=32, out_dim=1):
        super().__init__()
        self.base = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            Sine(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, out_dim),
        )
        self._init_with_noise()

    def _init_with_noise(self, std=0.5):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)
                nn.init.normal_(module.bias, mean=0.0, std=std)

    def forward(self, x):
        return self.base(x)


class SpectralGaussianFieldNoise(nn.Module):
    def __init__(self, grid_size=256, domain_size=20.0, beta=3.5):
        super().__init__()
        self.grid_size = int(grid_size)
        self.domain_size = float(domain_size)
        self.beta = float(beta)
        self.register_buffer(
            "field", torch.zeros((self.grid_size, self.grid_size), dtype=torch.float32)
        )
        self.resample()

    def resample(self):
        kx = np.fft.fftfreq(self.grid_size) * self.grid_size
        ky = np.fft.fftfreq(self.grid_size) * self.grid_size
        kx, ky = np.meshgrid(kx, ky, indexing="xy")
        k = np.sqrt(kx**2 + ky**2)

        # Isotropic power spectrum with beta controlling smoothness.
        power = (1.0 + k**2) ** (-self.beta / 2.0)
        power[0, 0] = 0.0

        coeff = np.random.normal(size=(self.grid_size, self.grid_size))
        coeff = coeff + 1j * np.random.normal(size=(self.grid_size, self.grid_size))
        coeff *= np.sqrt(power / 2.0)

        field = np.fft.ifft2(coeff).real
        field = (field - field.mean()) / (field.std() + 1e-8)
        self.field.copy_(torch.from_numpy(field.astype(np.float32)))

    def forward(self, x):
        # Periodic coordinate mapping onto the inverse-FFT grid.
        u = ((x[:, 0] + self.domain_size / 2.0) / self.domain_size) % 1.0
        v = ((x[:, 1] + self.domain_size / 2.0) / self.domain_size) % 1.0

        ix = torch.floor(u * self.grid_size).long() % self.grid_size
        iy = torch.floor(v * self.grid_size).long() % self.grid_size

        values = self.field[iy, ix]
        return values.unsqueeze(-1)


model = SpectralGaussianFieldNoise(grid_size=256, domain_size=20.0, beta=3.5)
model.eval()

SpectralGaussianFieldNoise()

In [7]:
def evaluate_on_grid(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x = np.linspace(x_range[0], x_range[1], resolution)
    y = np.linspace(y_range[0], y_range[1], resolution)
    xx, yy = np.meshgrid(x, y)
    grid = np.stack([xx.flatten(), yy.flatten()], axis=-1)

    with torch.no_grad():
        inputs = torch.from_numpy(grid).float()
        outputs = model(inputs).numpy().reshape(resolution, resolution)

    return x, y, xx, yy, outputs


def plot_2d_3d(model, x_range=(-10, 10), y_range=(-10, 10), resolution=120):
    x, y, xx, yy, outputs = evaluate_on_grid(
        model, x_range=x_range, y_range=y_range, resolution=resolution
    )

    # 2D heatmap and 3D surface side-by-side
    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "heatmap"}, {"type": "surface"}]],
        subplot_titles=("2D Heatmap", "3D Surface"),
        horizontal_spacing=0.08,
    )

    fig.add_trace(
        go.Heatmap(
            z=outputs, x=x, y=y, colorscale="Viridis", colorbar=dict(title="output")
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Surface(z=outputs, x=xx, y=yy, colorscale="Viridis", showscale=False),
        row=1,
        col=2,
    )

    fig.update_xaxes(title_text="x", row=1, col=1)
    fig.update_yaxes(title_text="y", row=1, col=1)
    fig.update_scenes(
        xaxis_title="x", yaxis_title="y", zaxis_title="output", row=1, col=2
    )
    fig.update_layout(
        height=520, width=1200, title=f"MLP output over {x_range} x {y_range}"
    )
    fig.show()


s = 10
r = (-s, s)
plot_2d_3d(model, x_range=r, y_range=r, resolution=120)